# WTI Oil Price Forecasting — One Agent, Three Tasks

> **Part 3 of 4.** This notebook builds on the agentic predictor introduced in
> [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb).

A single Analyst Agent — backed by bounded Google Search — answers three tasks
using **one system prompt** and **task-specific user payloads**:

| Stream | Task | Output |
|--------|------|--------|
| A | Trajectory | 5/10/21-day price forecasts |
| B | Binary shock | P(WTI +$5 in 5 days) |
| C | Scenario analysis | Top 3 expert scenarios for 60 days |


In [1]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


warnings.filterwarnings("ignore")

# ── Suppress spurious OpenTelemetry async-context errors ─────────────────────
# openinference auto-instruments litellm with OTel tracing. In Jupyter's event
# loop, asyncio Tasks are sometimes GC'd while pending, which throws GeneratorExit
# into OTel's context manager and causes a harmless but noisy ValueError when the
# context token tries to detach from a different asyncio Context. Patch it out.
from opentelemetry import context as _otel_ctx

_orig_detach = _otel_ctx.detach


def _safe_detach(token):  # type: ignore[no-untyped-def]
    try:
        _orig_detach(token)
    except ValueError:
        pass


_otel_ctx.detach = _safe_detach

# ── Model selection ───────────────────────────────────────────────────────────
# Override with a cheaper model for development; use "gemini-3.5-flash" for
# production-quality results committed to the cache.
AGENT_MODEL = "gemini-3.1-flash-lite"

from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_forecasting.analysis import compute_brier_score, trajectory_mae_table
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service, naive_utc_now
from energy_oil_forecasting.paths import (
    CLR_AGENT,
    CLR_PROPHET,
    PROPHET_SHOCK_TRAJ_CACHE,
    PROPHET_TRAJ_CACHE,
    SCENARIO_CACHE,
    SCENARIO_ORIGIN,
    SHOCK_ANALYST_CACHE,
    SHOCK_HORIZON,
    SHOCK_ORIGINS,
    SHOCK_THRESHOLD,
    TRAJ_AGENT_CACHE,
    TRAJECTORY_ORIGINS,
)
from energy_oil_forecasting.prophet_baseline import (
    check_shock_outcome,
    load_prophet_trajectories,
    prophet_prob_shock,
    wti_series_to_price_df,
)
from energy_oil_forecasting.tasks import TASK_SPECS, build_wti_news_predictor
from energy_oil_forecasting.viz import (
    conf_bar,
    make_shock_comparison_chart,
    make_trajectory_fan_chart,
    prob_bar,
    verdict_label,
)


data_service = build_wti_service()
ctx = data_service.context(as_of=naive_utc_now())
price_df = wti_series_to_price_df(ctx.get_series(WTI_SERIES_ID))

prophet_traj_df = load_prophet_trajectories(price_df, TRAJECTORY_ORIGINS, PROPHET_TRAJ_CACHE)
prophet_shock_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(f"Price history through {price_df.index[-1].date()}")

/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


Loaded 63 Prophet trajectory rows from energy_prophet_trajectories.parquet
Loaded 126 Prophet trajectory rows from energy_shock_prophet_trajectories.parquet
Price history through 2026-05-21


---
## Stream 1 — Trajectory Forecast

Compare Prophet fan charts to the news-grounded agent at three origins.

In [2]:
trajectory_task = ForecastingTask(
    task_id="wti_trajectory_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[5, 10, 21],
    frequency="B",
    description="Trajectory demo for NB3",
)

traj_predictor = build_wti_news_predictor("trajectory", model=AGENT_MODEL)

if TRAJ_AGENT_CACHE.exists():
    with open(TRAJ_AGENT_CACHE) as f:
        traj_agent_results = json.load(f)
    print(f"Loaded {len(traj_agent_results)} cached trajectory agent runs.")
else:
    traj_agent_results = []
    for origin in TRAJECTORY_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = traj_predictor.predict(trajectory_task, origin_ctx)
        traj_agent_results.append(
            {
                "origin": str(origin.date()),
                "predictions": [p.model_dump(mode="json") for p in preds],
            }
        )
    with open(TRAJ_AGENT_CACHE, "w") as f:
        json.dump(traj_agent_results, f, indent=2)
    print(f"Saved {len(traj_agent_results)} agent trajectory runs.")

# Summary: agent point forecasts at each origin
print("\nAgent trajectory summary:")
for r in traj_agent_results:
    preds = r["predictions"]
    pts = [f"h{[5,10,21][i]}=${preds[i]['payload']['point_forecast']:.1f}" for i in range(len(preds))]
    origin_price_rows = price_df[price_df.index >= pd.Timestamp(r["origin"])]
    origin_price = f"WTI=${origin_price_rows.iloc[0]['price']:.2f}" if not origin_price_rows.empty else ""
    print(f"  {r['origin']}  {origin_price}  {' | '.join(pts)}")

10:33:28 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
10:33:28 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Loaded 3 cached trajectory agent runs.

Agent trajectory summary:
  2026-02-02  WTI=$62.14  h5=$65.8 | h10=$65.0 | h21=$63.5
  2026-02-23  WTI=$66.31  h5=$67.5 | h10=$67.0 | h21=$65.0
  2026-03-02  WTI=$71.23  h5=$90.0 | h10=$94.0 | h21=$98.0


In [3]:
# ── I/O inspection: 2026-03-02 — conflict onset, most informative ────────────
INSPECT_ORIGIN = "2026-03-02"
inspect_rec = next((r for r in traj_agent_results if r["origin"] == INSPECT_ORIGIN), None)

if inspect_rec:
    origin_ts = pd.Timestamp(INSPECT_ORIGIN)
    bday_dates = pd.bdate_range(start=origin_ts + pd.offsets.BDay(1), periods=21)
    origin_price_row = price_df[price_df.index >= origin_ts]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")

    preds = inspect_rec["predictions"]
    rationale = preds[0].get("metadata", {}).get("agent_rationale", "") if preds else ""

    table_rows = "| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |\n|---|---|---|---|---|---|\n"
    for i, h in enumerate([5, 10, 21]):
        actual_rows = price_df[price_df.index >= bday_dates[h - 1]]
        actual = float(actual_rows.iloc[0]["price"]) if not actual_rows.empty else float("nan")
        pt = preds[i]["payload"]["point_forecast"]
        q10_val = next((v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.1) < 1e-6), float("nan"))
        q90_val = next((v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.9) < 1e-6), float("nan"))
        p_row = prophet_traj_df[(prophet_traj_df["origin"] == origin_ts) & (prophet_traj_df["horizon"] == h)]
        p_yhat = float(p_row.iloc[0]["yhat"]) if not p_row.empty else float("nan")
        table_rows += (
            f"| {h} bdays | **${pt:.1f}** | [{q10_val:.1f} – {q90_val:.1f}] "
            f"| ${actual:.1f} | {pt - actual:+.1f} | {p_yhat - actual:+.1f} |\n"
        )

    display(Markdown(
        f"### Stream 1 — I/O Inspection: {INSPECT_ORIGIN}  (WTI ${origin_price:.2f}/bbl)\n\n"
        "Agent and Prophet point forecasts vs realised prices at each horizon.\n\n"
        + table_rows
        + (f"\n> **Agent rationale:** {rationale}" if rationale else "")
    ))

### Stream 1 — I/O Inspection: 2026-03-02  (WTI $71.23/bbl)

Agent and Prophet point forecasts vs realised prices at each horizon.

| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |
|---|---|---|---|---|---|
| 5 bdays | **$90.0** | [77.0 – 109.0] | $94.8 | -4.8 | -30.2 |
| 10 bdays | **$94.0** | [76.0 – 120.0] | $93.5 | +0.5 | -29.2 |
| 21 bdays | **$98.0** | [75.0 – 135.0] | $101.4 | -3.4 | -36.9 |

> **Agent rationale:** Over the weekend of February 28, 2026, the Middle East descended into a severe direct military conflict. The joint U.S.-Israeli strikes on Iranian leadership and strategic sites, followed by Iran's retaliatory attacks and blockade of the Strait of Hormuz, have permanently altered the crude oil supply outlook, rendering previous bearish baseline forecasts (such as J.P. Morgan's Brent $60 average or EIA's WTI $51.42 average) obsolete. When futures markets resume trading on March 2, WTI (which closed at $65.21 on Feb 26 and $67.02 on Feb 27) will see an epic, panic-driven gap up. Our probabilistic forecasts reflect this massive structural breakout, modeling a rapid elevation in median prices to $90.00 (H5), $94.00 (H10), and $98.00 (H21). The distribution is highly skewed to the right to capture the severe tail-risk of a prolonged blockade and infrastructure damage (up to $150/bbl at the 95th percentile), while acknowledging a wider variance down to $70/bbl should the U.S. Navy successfully open the shipping lanes and defuse the physical supply bottleneck.

In [4]:
# ── Trajectory fan chart: Prophet fan vs agent error bars at 3 origins ───────
fig = make_trajectory_fan_chart(
    traj_agent_results, prophet_traj_df, price_df, TRAJECTORY_ORIGINS
)
fig.show()

# ── MAE evaluation table ──────────────────────────────────────────────────────
mae_df = trajectory_mae_table(traj_agent_results, prophet_traj_df, price_df)
if not mae_df.empty:
    display(mae_df.drop(columns=["Prophet MAE", "Agent MAE"]))
    mean_mae = mae_df[["Prophet MAE", "Agent MAE"]].mean()
    print(f"\nMean MAE  Prophet: ${mean_mae['Prophet MAE']:.2f}  Agent: ${mean_mae['Agent MAE']:.2f}")

Actual ($) Prophet ($) Agent ($)
Origin     Horizon                                  
2026-02-02 5 bdays        64.4        62.6      65.8
           10 bdays       62.3        63.4      65.0
           21 bdays       74.6        65.2      63.5
2026-02-23 5 bdays        71.2        64.6      67.5
           10 bdays       94.8        64.7      67.0
           21 bdays       92.3        64.4      65.0
2026-03-02 5 bdays        94.8        64.6      90.0
           10 bdays       93.5        64.3      94.0
           21 bdays      101.4        64.5      98.0


Mean MAE  Prophet: $19.23  Agent: $9.19


---
## Stream 2 — Binary Shock Prediction

In [5]:
shock_task = ForecastingTask(
    task_id="wti_upshock_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[SHOCK_HORIZON],
    frequency="B",
    description="Binary upshock demo",
)

shock_predictor = build_wti_news_predictor("shock", model=AGENT_MODEL)

if SHOCK_ANALYST_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_results = json.load(f)
    print(f"Loaded {len(shock_results)} cached shock forecasts.")
else:
    shock_results = []
    for origin in SHOCK_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = shock_predictor.predict(shock_task, origin_ctx)
        outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
        shock_results.append(
            {
                "origin": str(origin.date()),
                "probability": preds[0].payload.probability,
                "outcome": outcome,
                "delta": delta,
                "metadata": preds[0].metadata,
            }
        )
    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_results, f, indent=2)

agent_probs = [r["probability"] for r in shock_results]
outcomes = [r["outcome"] for r in shock_results]
print(f"Agent Brier score: {compute_brier_score(agent_probs, outcomes):.4f}")
print(f"Task spec preview:\n{TASK_SPECS['shock'][:200]}...")

Task was destroyed but it is pending!
task: <Task pending name='Task-45' coro=<<async_generator_athrow without __name__>()>>
Failed to detach context
Traceback (most recent call last):
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 589, in use_span
    yield span
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/openinference/instrumentation/_tracers.py", line 141, in start_as_current_span
    yield cast(OpenInferenceSpan, current_span)
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self.

Agent Brier score: 0.1992
Task spec preview:
Estimate P(up) — the probability that WTI will close MORE THAN
$5/bbl HIGHER than today's price at the end of
5 trading days.

Return JSON with exactly these fields:
{
  "probability": <float 0-1>,
  ...


In [6]:
# ── Per-origin forecast cards ─────────────────────────────────────────────────
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    label = origin.strftime("%b %-d, %Y")
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    a_prob = float(r["probability"])
    outcome = int(r["outcome"])
    delta = float(r["delta"])
    brier = (a_prob - outcome) ** 2
    meta = r.get("metadata", {})
    reasoning = meta.get("reasoning", meta.get("agent_rationale", "—"))
    key_signals = meta.get("key_signals", [])
    confidence = meta.get("confidence", "?")
    outcome_badge = "**SHOCK**" if outcome else "No shock"

    display(Markdown(
        f"---\n"
        f"### {label} — WTI ${origin_price:.2f}/bbl\n\n"
        f"| | |\n|---|---|\n"
        f"| **Prediction** | P(up > +${SHOCK_THRESHOLD:.0f}) = **{a_prob:.0%}**  `{prob_bar(a_prob)}` |\n"
        f"| **Confidence** | {confidence.title() if isinstance(confidence, str) else confidence}  {conf_bar(str(confidence))} |\n"
        f"| **Rationale** | {reasoning} |\n"
        f"| **Key signals** | {' · '.join(key_signals) if key_signals else '—'} |\n"
        f"| **Actual outcome** | {outcome_badge} — price moved **{delta:+.2f}/bbl** |\n"
        f"| **Verdict** | {verdict_label(a_prob, outcome, delta, SHOCK_THRESHOLD)} |\n"
        f"| **Brier score** | {brier:.3f} {'🟢' if brier < 0.10 else '🟡' if brier < 0.25 else '🔴'} |\n"
    ))

---
### Feb 2, 2026 — WTI $62.14/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | A $5/bbl move in 5 trading days represents an approximate 7.6% increase, a significant volatility event that typically requires a major exogenous supply disruption or geopolitical shock. Current market dynamics do not signal such an imminent event, as price action over the last few months has been characterized by sideways consolidation and moderate volatility. Consequently, while a sharp upside is theoretically possible, the probability of it occurring within such a short, defined window is relatively low. |
| **Key signals** | Current WTI price near $65.42 support/resistance zone · Lack of immediate, high-impact geopolitical or supply shock catalysts in recent data history · Volatility in short-term price movements suggests consolidation rather than a strong trend breach |
| **Actual outcome** | No shock — price moved **+2.22/bbl** |
| **Verdict** | Actual: +$2.22/bbl — no shock |
| **Brier score** | 0.022 🟢 |


---
### Feb 9, 2026 — WTI $64.36/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **15%**  `██░░░░░░░░  15%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | While WTI has shown some upward momentum, the requirement for a >$5/bbl increase in just 5 trading days is a significant "shock" move, which is statistically rare outside of acute supply chain disruptions or major geopolitical events. As of February 8, 2026, markets are characterized by a tension between steady global supply growth and simmering geopolitical risks. A 15% probability reflects the potential for sudden news-driven volatility, but without a confirmed immediate supply-chain collapse, the base case remains a more contained, incremental price movement. |
| **Key signals** | Escalating geopolitical risk premium in the Middle East · Strong momentum and tight range-trading leading into mid-February 2026 · Strong seasonality and ongoing supply constraints in early 2026 |
| **Actual outcome** | No shock — price moved **-2.03/bbl** |
| **Verdict** | Actual: +$-2.03/bbl — no shock |
| **Brier score** | 0.022 🟢 |


---
### Feb 16, 2026 — WTI $62.33/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **5%**  `░░░░░░░░░░  5%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | While geopolitical tensions in the Strait of Hormuz provide potential for sudden price spikes, the current market environment is characterized by significant fundamental resistance. With the IEA forecasting a substantial global supply surplus of ~1 million bpd and OPEC+ maintaining cuts to balance this, a rapid $5+/bbl increase in just 5 trading days would require a major, unexpected escalation, such as a physical disruption of supply, which is not currently the base case given ongoing diplomatic negotiations. Therefore, the probability of an 'upshock' of that magnitude remains low. |
| **Key signals** | Geopolitical risk premium associated with Iran/Strait of Hormuz tension · OPEC+ output cuts through March 2026 provide support floor · Projected global supply surplus creating a cap on significant price upside |
| **Actual outcome** | No shock — price moved **+3.98/bbl** |
| **Verdict** | Actual: +$3.98/bbl — no shock |
| **Brier score** | 0.003 🟢 |


---
### Feb 23, 2026 — WTI $66.31/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **35%**  `████░░░░░░  35%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | WTI is currently experiencing a significant bullish momentum driven by geopolitical tensions in the Persian Gulf and a tight inventory environment. While the probability of a $5/bbl move (to >$71.43) in just five days is inherently low, the threat of an imminent blockade of the Strait of Hormuz or actual military strikes provides a non-negligible catalyst for a sharp, panic-driven price surge. However, we balance this against the risk-off pressure from new U.S. import tariffs and the potential for short-term diplomatic cooling, which prevents us from assigning a higher probability to such a large, rapid gain. |
| **Key signals** | Heightened geopolitical risk (Iran/Strait of Hormuz threat) creates a massive war premium potential. · Tight prompt supplies following a 9 million-barrel inventory draw. · Upcoming 10-15 day deadline for Iranian nuclear negotiations increases likelihood of short-term volatility and potential price spikes. · Introduction of 15% import tariffs creates macroeconomic uncertainty/risk-off potential. |
| **Actual outcome** | No shock — price moved **+4.92/bbl** |
| **Verdict** | Actual: +$4.92/bbl — no shock |
| **Brier score** | 0.122 🟡 |


---
### Mar 2, 2026 — WTI $71.23/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **5%**  `░░░░░░░░░░  5%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | A $5/bbl increase represents a move of roughly 7.7% from the current price level of $65.21. Historically, such significant, rapid upside movements for WTI are typically driven by major, unforeseen supply shocks or extreme geopolitical events, which are not currently indicated by market data. Given the recent consolidation and lack of high volatility signals, the probability of achieving this specific threshold within just 5 trading days is assessed as very low. |
| **Key signals** | WTI price currently trading in a consolidation range near $65/bbl. · Lack of immediate geopolitical or supply-side catalysts to drive a rapid $5/bbl move in 5 trading days. · Statistical infrequency of >7.5% price spikes over short 5-day windows in stable market regimes. |
| **Actual outcome** | **SHOCK** — price moved **+23.54/bbl** |
| **Verdict** | Actual: +$23.54/bbl (>5) — shock materialised |
| **Brier score** | 0.902 🔴 |


---
### Mar 9, 2026 — WTI $94.77/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **35%**  `████░░░░░░  35%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | The market is currently reacting to a catastrophic supply chain shock caused by the closure of the Strait of Hormuz, which has already pushed prices significantly higher. While the momentum is extremely bullish, a further $5/bbl move within 5 days is a high bar, as it would require the conflict to escalate even further without a effective supply response or a massive, immediate SPR release to dampen volatility. I estimate a 35% probability for such an extreme upward shock, balancing the immediate supply scarcity panic against the strong likelihood of aggressive U.S. and global policy intervention to curb further price spikes. |
| **Key signals** | Strait of Hormuz closure disrupting 20% of global oil supply · WTI rally momentum triggered by sudden geopolitical conflict and production shutdowns · Potential US SPR release and Navy escorts as market-stabilizing counter-factors |
| **Actual outcome** | No shock — price moved **-1.27/bbl** |
| **Verdict** | Actual: +$-1.27/bbl — no shock |
| **Brier score** | 0.122 🟡 |


In [7]:
# ── Prophet probabilities for the shock origins ───────────────────────────────
prophet_shock_probs = []
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    p_sub = prophet_shock_df[prophet_shock_df["origin"] == origin]
    prophet_shock_probs.append(prophet_prob_shock(p_sub, origin_price, SHOCK_THRESHOLD, SHOCK_HORIZON))

# ── Comparison chart: P(shock) over time + cumulative Brier ──────────────────
fig = make_shock_comparison_chart(shock_results, prophet_shock_probs, shock_threshold=SHOCK_THRESHOLD)
fig.show()

# ── Brier score summary ───────────────────────────────────────────────────────
agent_probs = [float(r["probability"]) for r in shock_results]
outcomes = [int(r["outcome"]) for r in shock_results]
agent_brier = compute_brier_score(agent_probs, outcomes)
valid_prophet = [(p, o) for p, o in zip(prophet_shock_probs, outcomes) if not np.isnan(p)]
prophet_brier = compute_brier_score([p for p, _ in valid_prophet], [o for _, o in valid_prophet])
brier_df = pd.DataFrame(
    {"Mean Brier score": [f"{agent_brier:.4f}", f"{prophet_brier:.4f}"]},
    index=pd.Index(["Analyst Agent", "Prophet"], name="Method"),
)
print("Mean Brier score (lower = better, 0.25 = random ceiling):")
display(brier_df)

Mean Brier score (lower = better, 0.25 = random ceiling):


,Mean Brier score
Method,
Analyst Agent,0.1992
Prophet,0.1927


---
## Stream 3 — Scenario Analysis


In [8]:
scenario_task = ForecastingTask(
    task_id="wti_scenario_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[21],
    frequency="B",
    description="Scenario analysis demo",
)

scenario_predictor = build_wti_news_predictor("scenario", model=AGENT_MODEL)

if SCENARIO_CACHE.exists():
    with open(SCENARIO_CACHE) as f:
        scenario_payload = json.load(f)
    print("Loaded cached scenario analysis.")
else:
    as_of = SCENARIO_ORIGIN - pd.Timedelta(days=1)
    origin_ctx = data_service.context(as_of=as_of)
    preds = scenario_predictor.predict(scenario_task, origin_ctx)
    scenario_payload = preds[0].metadata
    with open(SCENARIO_CACHE, "w") as f:
        json.dump(scenario_payload, f, indent=2)

# ── Rich scenario cards ───────────────────────────────────────────────────────
scenario_origin_price_row = price_df[price_df.index >= SCENARIO_ORIGIN]
scenario_origin_price = float(scenario_origin_price_row.iloc[0]["price"]) if not scenario_origin_price_row.empty else float("nan")

display(Markdown(
    f"#### Stream 3 — Scenario Analysis  "
    f"*(origin: {SCENARIO_ORIGIN.date()}, WTI ${scenario_origin_price:.2f}/bbl)*\n\n"
    f"Base case: **{scenario_payload.get('base_case', '?')}**"
))

base_case = scenario_payload.get("base_case", "")
for s in scenario_payload.get("scenarios", []):
    name = s.get("name", "?")
    desc = s.get("description", "")
    prob = float(s.get("probability", 0))
    rng = s.get("wti_range_60d", [float("nan"), float("nan")])
    lo_r, hi_r = float(rng[0]), float(rng[1])
    pe = float(s.get("point_estimate_60d", float("nan")))
    drivers = s.get("key_drivers", [])
    base_marker = "  \u2605 **base case**" if name == base_case else ""

    display(Markdown(
        f"---\n"
        f"**{name}**{base_marker}\n\n"
        f"{desc}\n\n"
        f"| | |\n|---|---|\n"
        f"| Probability | **{prob:.0%}**  `{prob_bar(prob)}` |\n"
        f"| WTI range (60 days) | ${lo_r:.0f} \u2013 ${hi_r:.0f} /bbl |\n"
        f"| Point estimate | **${pe:.0f} /bbl** |\n"
        f"| Key drivers | {' \u00b7 '.join(drivers) if drivers else '\u2014'} |\n"
    ))

overall = scenario_payload.get("reasoning", "")
if overall:
    display(Markdown(f"---\n\n> **Overall reasoning:** {overall}"))

Task was destroyed but it is pending!
task: <Task pending name='Task-194' coro=<<async_generator_athrow without __name__>()>>
Failed to detach context
Traceback (most recent call last):
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 589, in use_span
    yield span
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/openinference/instrumentation/_tracers.py", line 141, in start_as_current_span
    yield cast(OpenInferenceSpan, current_span)
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 155, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/Users/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/opentelemetry/context/contextvars_context.py", line 53, in detach
    self

#### Stream 3 — Scenario Analysis  *(origin: 2026-03-02, WTI $71.23/bbl)*

Base case: **Geopolitical Risk Premium with Tepid Stabilization**

---
**Supply Disruption Escalation**

Conflict escalates; Strait of Hormuz remains closed or heavily restricted for the full 60 days, overwhelming any SPR releases and forcing a sustained, high-price environment.

| | |
|---|---|
| Probability | **30%**  `███░░░░░░░  30%` |
| WTI range (60 days) | $85 – $110 /bbl |
| Point estimate | **$95 /bbl** |
| Key drivers | Prolonged blockade of Strait of Hormuz · Failure of diplomatic interventions · Supply shortfall exceeding reserve releases |


---
**Geopolitical Risk Premium with Tepid Stabilization**  ★ **base case**

Geopolitical tensions persist but stabilize due to aggressive SPR releases and effective naval patrols ensuring tanker security, leading to a moderate, range-bound price environment.

| | |
|---|---|
| Probability | **50%**  `█████░░░░░  50%` |
| WTI range (60 days) | $68 – $88 /bbl |
| Point estimate | **$78 /bbl** |
| Key drivers | Coordinated SPR releases · Stabilization of tanker security via naval presence · OPEC+ production maintenance |


---
**Rapid De-escalation and Normalization**

De-escalation begins; rapid diplomatic breakthroughs lead to the reopening of the Strait, causing an immediate, sharp collapse in the risk premium built into prices.

| | |
|---|---|
| Probability | **20%**  `██░░░░░░░░  20%` |
| WTI range (60 days) | $55 – $70 /bbl |
| Point estimate | **$62 /bbl** |
| Key drivers | Diplomatic breakthrough in regional conflict · Immediate reopening of the Strait of Hormuz · Market perception of surplus return |


---

## Summary

One agent identity (`build_wti_multitask_news_config` / `build_wti_news_config`) with
three task-specific prompt builders and output schemas demonstrates the bootcamp
pattern for multi-task agentic forecasting. Continue to
[`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb) for the
production backtest harness.
